# Classificação com Support Vector Machines (SVM)
**Isabelli Reno** **|** **11241100124**

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)] (https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)

**Objetivo:** Construir um modelo de classificação binária com SVM para diagnósticar tumores mamários em Malignos (M) ou Benignos (B).

In [1]:
# Importação de bibliotecas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style = "whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Carregamento e limpeza inicial dos dados

In [2]:
## import os

filepath = "/kaggle/input/datasets/uciml/breast-cancer-wisconsin-data/data.csv"

# Verificação de diretório
if not os.path.exists(filepath):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            # Imprime apenas o arquivo .csv
            if filename.endswith('.csv'):
                filepath = os.path.join(dirname, filename)
                break # Para se corresponder

print(f'Arquivo de dados: {filepath}')

# Leitura
try:
    df = pd.read_csv(filepath, encoding = 'utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(filepath, encoding = 'latin1')

# Remove a coluna ID (caso exista)
if 'id'in df.columns:
    df = df.drop(columns =['id'])

# Remove colunas vazias
df = df.dropna(how = 'all', axis = 1)

# M = 1 (maligno), B = 0 (benigno)
df['diagnosis_encoded'] = df['diagnosis'].map({'M': 1, 'B': 0})

print('Dimensão do dataset:', df.shape)
print('\nDistribuição dos diagnósticos:')
print(df['diagnosis'].value_counts())
df.head()

NameError: name 'os' is not defined

## 2. Divisão e padronização dos dados

In [ ]:
# Separação das Características e Rótulos
X = df.drop(columns = ['diagnosis', 'diagnosis_encoded'])
y = df['diagnosis_encoded']

# Divisão de Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

print(f'Formato X treino: {X_train.shape}')
print(f'Formato X teste: {X_test.shape}')

# Distribuição de classes
plt.figure(figsize = (6, 4))
sns.countplot(data = df, hue = 'diagnosis', x = 'diagnosis', palette = 'Set1', legend = False)
plt.title('Distribuição dos Diagnósticos de Câncer de Mama')
plt.xlabel('Diagnóstico (M = Maligno, B = Benigno)')
plt.ylabel('Contagem')
plt.show

## 3. Modelagem SVM (Linear x RBF)

In [ ]:
# SVM Linear
pipe_linear = Pipeline([
    ('scalter', StandardScaler()),
    ('svm', SVC(kernel = 'linear', C = 1.0, random_state = 42))
])

pipe_linear.fit(X_train, y_train)
y_pred_linear = pipe_linear.predict(X_test)

# SVM Não Linear com Kernel RBF
pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel = 'rbf', C = 1.0, gamma = 'scale', random_state = 42))
])

pipe_rbf.fit(X_train, y_train)
y_pred_rbf = pipe_rbf.predict(X_test)

print(f'Precisão SVM Linear: {accuracy_score(y_test, y_pred_linear):.4f}')
print(f'Precisão do SVM RBF: {accuracy_score(y_test, y_pred_rbf):.4f}')

## 4. Ajuste de Hiperparâmetros

In [ ]:
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1]
}

grid_search = GridSearchCV(pipe_rbf, param_grid, cv = 5, scoring = 'accuracy')
grid_search.fit(X_train, y_train)

print('Melhores hiperparâmetros encontrados:')
print(grid_search.best_params_)

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

## 5. Avaliação de Desempenho

In [ ]:
print('Relatório do Melhor Modelo:\n')
print(classification_report(y_test, y_pred_best, target_names = ['Benigno (B)', 'Maligno (M)']))

# Matriz de confusão
fig, ax = plt.subplots(figsize = (6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels = ['Benigno (B)', 'Maligno (M)'],
    cmap = 'Blues', ax = ax
)

plt.title('Matriz de Confusão - SVM')
plt.show()